In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test_data.txt to test_data (1).txt
Saving train_data.txt to train_data (1).txt


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
    recall_score,
    precision_score,
    f1_score
)

In [ ]:
train_file = [f for f in uploaded.keys() if "train" in f.lower()][0]
test_file  = [f for f in uploaded.keys() if "test" in f.lower()][0]

train_df = pd.read_csv(train_file, header=None)
test_df  = pd.read_csv(test_file, header=None)

print("Train:", train_file, train_df.shape)
print("Test :", test_file, test_df.shape)

Train: train_data (1).txt (1040, 29)
Test : test_data (1).txt (168, 28)


In [ ]:
feature_names = [
    "Jitter_local", "Jitter_abs", "Jitter_rap", "Jitter_ppq5", "Jitter_ddp",
    "Shimmer_local", "Shimmer_dB", "Shimmer_apq3", "Shimmer_apq5", "Shimmer_apq11", "Shimmer_dda",
    "AC", "NTH", "HTN",
    "Median_pitch", "Mean_pitch", "Std_pitch", "Min_pitch", "Max_pitch",
    "Num_pulses", "Num_periods", "Mean_period", "Std_period",
    "Frac_unvoiced_frames", "Num_voice_breaks", "Degree_voice_breaks"
]

train_df.columns = ["SubjectID"] + feature_names + ["UPDRS", "Class"]
test_df.columns  = ["SubjectID"] + feature_names + ["Class"]

train_df["SampleNo"] = train_df.groupby("SubjectID").cumcount() + 1
test_df["SampleNo"]  = test_df.groupby("SubjectID").cumcount() + 1

print(train_df.head())
print(test_df.head())

   SubjectID  Jitter_local  Jitter_abs  Jitter_rap  Jitter_ppq5  Jitter_ddp  \
0          1         1.488    0.000090       0.900        0.794       2.699   
1          1         0.728    0.000038       0.353        0.376       1.059   
2          1         1.220    0.000074       0.732        0.670       2.196   
3          1         2.502    0.000123       1.156        1.634       3.469   
4          1         3.509    0.000167       1.715        1.539       5.145   

   Shimmer_local  Shimmer_dB  Shimmer_apq3  Shimmer_apq5  ...  Num_pulses  \
0          8.334       0.779         4.517         4.609  ...         160   
1          5.864       0.642         2.058         3.180  ...         170   
2          8.719       0.875         4.347         5.166  ...        1431   
3         13.513       1.273         5.263         8.771  ...          94   
4          9.112       1.040         3.102         4.927  ...         117   

   Num_periods  Mean_period  Std_period  Frac_unvoiced_frames 

In [ ]:
train_df["Vowel"] = train_df["SampleNo"].map({
    1: "a",
    2: "o",
    3: "u"
})

test_df["Vowel"] = test_df["SampleNo"].map({
    1: "a",
    2: "a",
    3: "a",
    4: "o",
    5: "o",
    6: "o"
})

train_vowel_df = train_df[train_df["Vowel"].isin(["a", "o", "u"])].copy()
test_vowel_df  = test_df[test_df["Vowel"].isin(["a", "o"])].copy()

print("Train vowel:", train_vowel_df.shape)
print("Test vowel :", test_vowel_df.shape)

Train vowel: (120, 31)
Test vowel : (168, 30)


In [ ]:
def add_robust_weights(data):
    data = data.copy()
    weights_all = []

    for sid, group in data.groupby("SubjectID"):
        Xg = group[feature_names]

        median = Xg.median()
        mad = (Xg - median).abs().median() + 1e-6

        robust_z = ((Xg - median) / mad).abs()
        score = robust_z.mean(axis=1)

        weights = np.exp(-score)

        weights_all.extend(weights.values)

    data["Weight"] = weights_all
    return data

In [ ]:
def make_weighted_subject_level(data):
    rows = []
    labels = []
    subject_ids = []

    for sid, group in data.groupby("SubjectID"):
        Xg = group[feature_names].values
        w = group["Weight"].values.reshape(-1, 1)

        w_sum = w.sum()

        wmean = (Xg * w).sum(axis=0) / w_sum
        wvar = (w * (Xg - wmean) ** 2).sum(axis=0) / w_sum
        wstd = np.sqrt(wvar)

        row = np.concatenate([wmean, wstd])

        rows.append(row)
        labels.append(group["Class"].iloc[0])
        subject_ids.append(sid)

    cols = [c + "_wmean" for c in feature_names] + [c + "_wstd" for c in feature_names]

    subject_df = pd.DataFrame(rows, columns=cols)
    subject_df["SubjectID"] = subject_ids
    subject_df["Class"] = labels

    return subject_df

In [ ]:
def evaluate_loo_weighted(X, y, k, C, gamma):
    loo = LeaveOneOut()

    y_true = []
    y_pred = []

    for train_idx, test_idx in loo.split(X):

        model = Pipeline([
            ("scaler", RobustScaler()),
            ("select", SelectKBest(
                score_func=mutual_info_classif,
                k=min(k, X.shape[1])
            )),
            ("svm", SVC(
                kernel="rbf",
                C=C,
                gamma=gamma,
                class_weight="balanced"
            ))
        ])

        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])

        y_true.append(y[test_idx][0])
        y_pred.append(pred[0])

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Sensitivity": recall_score(y_true, y_pred),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "CM": cm
    }

In [ ]:
vowel_sets = [
    ["a"],
    ["o"],
    ["u"],
    ["a", "o"],
    ["a", "u"],
    ["o", "u"],
    ["a", "o", "u"]
]

k_values = [5, 10, 15, 20, 30, 40, 52]
C_values = [1, 10, 100]
gamma_values = [0.01, 0.001]

results = []

for vowels in vowel_sets:
    print("Çalışıyor:", vowels)

    temp = train_vowel_df[train_vowel_df["Vowel"].isin(vowels)].copy()

    temp = add_robust_weights(temp)
    subject_df = make_weighted_subject_level(temp)

    X = subject_df.drop(columns=["SubjectID", "Class"]).values
    y = subject_df["Class"].values

    for k in k_values:
        for C in C_values:
            for gamma in gamma_values:

                res = evaluate_loso_weighted(X, y, k, C, gamma)

                results.append({
                    "Vowels": "+".join(vowels),
                    "K": k,
                    "C": C,
                    "gamma": gamma,
                    **res
                })

results_df = pd.DataFrame(results)
results_df.sort_values(by="MCC", ascending=False).head(20)

Çalışıyor: ['a']
Çalışıyor: ['o']
Çalışıyor: ['u']
Çalışıyor: ['a', 'o']
Çalışıyor: ['a', 'u']
Çalışıyor: ['o', 'u']
Çalışıyor: ['a', 'o', 'u']


,Vowels,K,C,gamma,Accuracy,Sensitivity,Specificity,Precision,F1,MCC,CM
19,a,20,1,0.001,1.000,1.00,1.00,1.000000,1.000000,1.000000,"[[20, 0], [0, 20]]"
49,o,10,1,0.001,1.000,1.00,1.00,1.000000,1.000000,1.000000,"[[20, 0], [0, 20]]"
1,a,5,1,0.001,1.000,1.00,1.00,1.000000,1.000000,1.000000,"[[20, 0], [0, 20]]"
43,o,5,1,0.001,1.000,1.00,1.00,1.000000,1.000000,1.000000,"[[20, 0], [0, 20]]"
13,a,15,1,0.001,1.000,1.00,1.00,1.000000,1.000000,1.000000,"[[20, 0], [0, 20]]"
7,a,10,1,0.001,0.975,1.00,0.95,0.952381,0.975610,0.951190,"[[19, 1], [0, 20]]"
67,o,30,1,0.001,0.975,1.00,0.95,0.952381,0.975610,0.951190,"[[19, 1], [0, 20]]"
55,o,15,1,0.001,0.975,1.00,0.95,0.952381,0.975610,0.951190,"[[19, 1], [0, 20]]"
85,u,5,1,0.001,0.950,1.00,0.90,0.909091,0.952381,0.904534,"[[18, 2], [0, 20]]"
25,a,30,1,0.001,0.950,1.00,0.90,0.909091,0.952381,0.904534,"[[18, 2], [0, 20]]"


In [ ]:
best = results_df.sort_values(by="MCC", ascending=False).iloc[0]

print("===== EN İYİ SONUÇ =====")
print("Vowels      :", best["Vowels"])
print("K           :", best["K"])
print("C           :", best["C"])
print("gamma       :", best["gamma"])
print("Accuracy    :", round(best["Accuracy"], 4))
print("Sensitivity :", round(best["Sensitivity"], 4))
print("Specificity :", round(best["Specificity"], 4))
print("Precision   :", round(best["Precision"], 4))
print("F1-score    :", round(best["F1"], 4))
print("MCC         :", round(best["MCC"], 4))
print("\nConfusion Matrix:")
print(best["CM"])

===== EN İYİ SONUÇ =====
Vowels      : a
K           : 20
C           : 1
gamma       : 0.001
Accuracy    : 1.0
Sensitivity : 1.0
Specificity : 1.0
Precision   : 1.0
F1-score    : 1.0
MCC         : 1.0

Confusion Matrix:
[[20  0]
 [ 0 20]]


In [ ]:
best_vowels = best["Vowels"].split("+")

test_available_vowels = ["a", "o"]

if any(v not in test_available_vowels for v in best_vowels):
    print("UYARI: En iyi LOSO modeli test setinde olmayan vowel içeriyor:", best_vowels)
    print("Test için sadece a, o veya a+o içeren en iyi modeli seçeceğiz.")

    test_compatible = results_df[
        results_df["Vowels"].isin(["a", "o", "a+o"])
    ].copy()

    best_test = test_compatible.sort_values(by="MCC", ascending=False).iloc[0]
else:
    best_test = best

print("===== TEST İÇİN SEÇİLEN MODEL =====")
print("Vowels:", best_test["Vowels"])
print("K:", best_test["K"])
print("C:", best_test["C"])
print("gamma:", best_test["gamma"])
print("Accuracy:", best_test["Accuracy"])
print(" MCC:", best_test["MCC"])

===== TEST İÇİN SEÇİLEN MODEL =====
Vowels: a
K: 20
C: 1
gamma: 0.001
Accuracy: 1.0
 MCC: 1.0


In [ ]:
final_vowels = best_test["Vowels"].split("+")
final_k = int(best_test["K"])
final_C = best_test["C"]
final_gamma = best_test["gamma"]

train_final = train_vowel_df[train_vowel_df["Vowel"].isin(final_vowels)].copy()
test_final  = test_vowel_df[test_vowel_df["Vowel"].isin(final_vowels)].copy()

train_final = add_robust_weights(train_final)
test_final  = add_robust_weights(test_final)

train_subject = make_weighted_subject_level(train_final)
test_subject  = make_weighted_subject_level(test_final)

X_train = train_subject.drop(columns=["SubjectID", "Class"]).values
y_train = train_subject["Class"].values

X_test = test_subject.drop(columns=["SubjectID", "Class"]).values
y_test = test_subject["Class"].values

final_model = Pipeline([
    ("scaler", RobustScaler()),
    ("select", SelectKBest(
        score_func=mutual_info_classif,
        k=min(final_k, X_train.shape[1])
    )),
    ("svm", SVC(
        kernel="rbf",
        C=final_C,
        gamma=final_gamma,
        class_weight="balanced"
    ))
])

final_model.fit(X_train, y_train)

test_pred = final_model.predict(X_test)

print("===== INDEPENDENT TEST RESULT =====")
print("Selected vowels:", final_vowels)
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Sensitivity / Recall:", round(recall_score(y_test, test_pred), 4))
print("F1-score:", round(f1_score(y_test, test_pred), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

===== INDEPENDENT TEST RESULT =====
Selected vowels: ['a']
Accuracy: 0.9643
Sensitivity / Recall: 0.9643
F1-score: 0.9818
Confusion Matrix:
[[ 0  0]
 [ 1 27]]


In [ ]:
selected_mask = final_model.named_steps["select"].get_support()
feature_cols = train_subject.drop(columns=["SubjectID", "Class"]).columns.tolist()

selected_features = np.array(feature_cols)[selected_mask]

print("===== SEÇİLEN ÖZELLİKLER =====")
for f in selected_features:
    print("-", f)

===== SEÇİLEN ÖZELLİKLER =====
- Jitter_local_wmean
- Jitter_ppq5_wmean
- Shimmer_local_wmean
- Shimmer_dB_wmean
- Shimmer_apq3_wmean
- Shimmer_apq11_wmean
- Shimmer_dda_wmean
- Median_pitch_wmean
- Mean_pitch_wmean
- Min_pitch_wmean
- Mean_period_wmean
- Frac_unvoiced_frames_wmean
- Degree_voice_breaks_wmean
- Jitter_rap_wstd
- Shimmer_apq5_wstd
- Median_pitch_wstd
- Min_pitch_wstd
- Mean_period_wstd
- Std_period_wstd
- Num_voice_breaks_wstd


In [ ]:
import joblib
import os

os.makedirs("model", exist_ok=True)

unlu_test_results = {
    "selected_vowels": final_vowels,
    "k": int(final_k),
    "C": float(final_C),
    "gamma": float(final_gamma),
    "y_true": y_test.tolist(),
    "y_pred": test_pred.tolist(),
    "confusion_matrix": confusion_matrix(y_test, test_pred).tolist(),
    "accuracy": float(accuracy_score(y_test, test_pred)),
    "sensitivity": float(recall_score(y_test, test_pred)),
    "f1": float(f1_score(y_test, test_pred)),
    "selected_features": selected_features.tolist()
}

joblib.dump(final_model, "model/unlu_final_model.pkl")
joblib.dump(train_subject, "model/train_subject_unlu.pkl")
joblib.dump(test_subject, "model/test_subject_unlu.pkl")
joblib.dump(unlu_test_results, "model/unlu_test_results.pkl")

print("✓ Ünlü odaklı model ve bağımsız test sonuçları kaydedildi.")

✓ Ünlü odaklı model ve bağımsız test sonuçları kaydedildi.


In [ ]:
from google.colab import files

files.download("model/unlu_final_model.pkl")
files.download("model/train_subject_unlu.pkl")
files.download("model/test_subject_unlu.pkl")
files.download("model/unlu_test_results.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>